<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/Python-Notebook-Banners/Exercise.png"  style="display: block; margin-left: auto; margin-right: auto;";/>
</div>

# Exercise: The random forest
© ExploreAI Academy

In this exercise, we build, evaluate, and compare random forest regression models.

## Learning objectives

By the end of this train, you should be able to:
* Build a random forest regression model in Python.
* Experiment with different numbers of trees.
* Evaluate feature importance using a random forest. 

## Exercises

In this exercise, we will be using the `Crop_yield` dataset which contains various factors that could influence the yield of a particular crop across different regions.

### Import libraries and dataset

In [54]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics

In [55]:
# Load dataset
df= pd.read_csv("https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Data/Python/Crop_yield.csv")
df.head(5)

,Region,Temperature,Rainfall,Soil_Type,Fertilizer_Usage,Pesticide_Usage,Irrigation,Crop_Variety,Yield
0,East,23.152156,803.362573,Clayey,204.792011,20.767590,1,Variety B,40.316318
1,West,19.382419,571.567670,Sandy,256.201737,49.290242,0,Variety A,26.846639
2,North,27.895890,-8.699637,Loamy,222.202626,25.316121,0,Variety C,-0.323558
3,East,26.741361,897.426194,Loamy,187.984090,17.115362,0,Variety C,45.440871
4,East,19.090286,649.384694,Loamy,110.459549,24.068804,1,Variety B,35.478118


In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Region            1000 non-null   object 
 1   Temperature       1000 non-null   float64
 2   Rainfall          1000 non-null   float64
 3   Soil_Type         1000 non-null   object 
 4   Fertilizer_Usage  1000 non-null   float64
 5   Pesticide_Usage   1000 non-null   float64
 6   Irrigation        1000 non-null   int64  
 7   Crop_Variety      1000 non-null   object 
 8   Yield             1000 non-null   float64
dtypes: float64(5), int64(1), object(3)
memory usage: 70.4+ KB


### Preparing the dataset

In the code below, we prepare our dataset for modelling by encoding categorical variables to convert them to a numeric format.

In [57]:
numeric_features = df.select_dtypes(include=[np.number])
categorical_features = df.select_dtypes(include=[object])

categorical_features

,Region,Soil_Type,Crop_Variety
0,East,Clayey,Variety B
1,West,Sandy,Variety A
2,North,Loamy,Variety C
3,East,Loamy,Variety C
4,East,Loamy,Variety B
...,...,...,...
995,North,Clayey,Variety C
996,North,Clayey,Variety C
997,West,Clayey,Variety A
998,West,Loamy,Variety C


In [58]:
# encode categorical features
categorical_features_encoded = pd.get_dummies(categorical_features, drop_first=False)

# convert encoded categorical to 0 and 1
categorical_features_encoded = categorical_features_encoded.astype(int)

df_encoded = pd.concat([numeric_features, categorical_features_encoded], axis=1)

df_encoded.head(5)

,Temperature,Rainfall,Fertilizer_Usage,Pesticide_Usage,Irrigation,Yield,Region_East,Region_North,Region_South,Region_West,Soil_Type_Clayey,Soil_Type_Loamy,Soil_Type_Sandy,Crop_Variety_Variety A,Crop_Variety_Variety B,Crop_Variety_Variety C
0,23.152156,803.362573,204.792011,20.767590,1,40.316318,1,0,0,0,1,0,0,0,1,0
1,19.382419,571.567670,256.201737,49.290242,0,26.846639,0,0,0,1,0,0,1,1,0,0
2,27.895890,-8.699637,222.202626,25.316121,0,-0.323558,0,1,0,0,0,1,0,0,0,1
3,26.741361,897.426194,187.984090,17.115362,0,45.440871,1,0,0,0,0,1,0,0,0,1
4,19.090286,649.384694,110.459549,24.068804,1,35.478118,1,0,0,0,0,1,0,0,1,0


In [59]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Temperature             1000 non-null   float64
 1   Rainfall                1000 non-null   float64
 2   Fertilizer_Usage        1000 non-null   float64
 3   Pesticide_Usage         1000 non-null   float64
 4   Irrigation              1000 non-null   int64  
 5   Yield                   1000 non-null   float64
 6   Region_East             1000 non-null   int64  
 7   Region_North            1000 non-null   int64  
 8   Region_South            1000 non-null   int64  
 9   Region_West             1000 non-null   int64  
 10  Soil_Type_Clayey        1000 non-null   int64  
 11  Soil_Type_Loamy         1000 non-null   int64  
 12  Soil_Type_Sandy         1000 non-null   int64  
 13  Crop_Variety_Variety A  1000 non-null   int64  
 14  Crop_Variety_Variety B  1000 non-null   i

### Exercise 1

Create a function named `train_rf_model` to train and evaluate a random forest regression model on the encoded dataset. 

The function should take in three parameters:
- A DataFrame containing the encoded features.
- A string containing the name of the target variable.
- The number of estimators for the random forest. 

It then returns: 
- The trained model object. 
- The RMSE and R<sup>2</sup> scores of the model's performance on the test set. 

In [60]:
def train_rf_model(df, target_column, n_estimators):
    """
    Train a Random Forest model on the given dataframe.

    Parameters:
    df (pd.DataFrame): The input dataframe containing features and target.
    target_column (str): The name of the target column in the dataframe.
    n_estimators (int): The number of trees in the forest.

    Returns:
    model (RandomForestRegressor): The trained Random Forest model.
    RMSE (float): The Root Mean Squared Error of the model on the test set.
    R_squared (float): The R-squared value of the model on the test set.
    """
    X = df.drop(columns=[target_column])
    y = df[target_column]

    # standardize the features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

    # create and train the Random Forest model
    model = RandomForestRegressor(n_estimators=n_estimators, random_state=42)

    model.fit(X_train, y_train)

    # make predictions on the test and training sets
    training_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # Training and Testing Residuals
    training_residuals = y_train - training_pred
    testing_residuals = y_test - test_pred

    # calculate RMSE and R-squared for the test and training sets
    training_rmse = np.sqrt(metrics.mean_squared_error(y_train, training_pred))
    testing_rmse = np.sqrt(metrics.mean_squared_error(y_test, test_pred))
    training_r_squared = metrics.r2_score(y_train, training_pred)
    testing_r_squared = metrics.r2_score(y_test, test_pred)
    training_mse = metrics.mean_squared_error(y_train, training_pred)
    testing_mse = metrics.mean_squared_error(y_test, test_pred)
    training_mae = metrics.mean_absolute_error(y_train, training_pred)
    testing_mae = metrics.mean_absolute_error(y_test, test_pred)
    training_mape = np.mean(np.abs(training_residuals / y_train)) * 100
    testing_mape = np.mean(np.abs(testing_residuals / y_test)) * 100
    training_rss = np.sum(training_residuals ** 2)
    testing_rss = np.sum(testing_residuals ** 2)

    # make table of metrics
    metrics_table = pd.DataFrame({
        'Metric': ['RMSE', 'R-squared', 'MSE', 'MAE', 'MAPE', 'RSS'],
        'Training': [training_rmse, training_r_squared, training_mse, training_mae, training_mape, training_rss],
        'Testing': [testing_rmse, testing_r_squared, testing_mse, testing_mae, testing_mape, testing_rss]
    })

    return model, testing_rmse, testing_r_squared, metrics_table

In [61]:
target_column = 'Yield'
n_estimators = 100

trained_model, test_rmse, test_r_squared, metrics_table = train_rf_model(df_encoded, target_column, n_estimators)

In [62]:
trained_model

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [63]:
test_rmse

np.float64(0.8552713754444304)

In [64]:
test_r_squared

0.9921019352462954

In [65]:
metrics_table

,Metric,Training,Testing
0,RMSE,0.356752,0.855271
1,R-squared,0.998802,0.992102
2,MSE,0.127272,0.731489
3,MAE,0.259364,0.653548
4,MAPE,1.615750,2.858417
5,RSS,101.817828,146.297825


### Exercise 2

Use the function you have defined in **Exercise 1** to train and evaluate three different random forest regression models with each having the following number of estimators respectively: `50`, `100`, and `200`. Store the results in a dictionary.

In [66]:
rf_1_trained_model, rf_1_test_rmse, rf_1_test_r_squared, rf_1_metrics_table = train_rf_model(df_encoded, target_column, 50)
rf_2_trained_model, rf_2_test_rmse, rf_2_test_r_squared, rf_2_metrics_table = train_rf_model(df_encoded, target_column, 100)
rf_3_trained_model, rf_3_test_rmse, rf_3_test_r_squared, rf_3_metrics_table = train_rf_model(df_encoded, target_column, 200)

In [67]:
rf_1 = [rf_1_trained_model, rf_1_test_rmse, rf_1_test_r_squared, rf_1_metrics_table]
rf_2 = [rf_2_trained_model, rf_2_test_rmse, rf_2_test_r_squared, rf_2_metrics_table]
rf_3 = [rf_3_trained_model, rf_3_test_rmse, rf_3_test_r_squared, rf_3_metrics_table]


for i, rf in enumerate([rf_1, rf_2, rf_3], start=1):
    print(f"Random Forest Model {i}:")
    print(f"Number of Estimators: {50 * (2 ** (i - 1))}")
    print(f"Test RMSE: {rf[1]}")
    print(f"Test R-squared: {rf[2]}")
    print("Metrics Table:")
    print(rf[3])
    print("=" * 70+"\n")


Random Forest Model 1:
Number of Estimators: 50
Test RMSE: 0.8555332568271099
Test R-squared: 0.9920970977810426
Metrics Table:
      Metric    Training     Testing
0       RMSE    0.359784    0.855533
1  R-squared    0.998782    0.992097
2        MSE    0.129445    0.731937
3        MAE    0.262448    0.659691
4       MAPE    1.584979    2.918050
5        RSS  103.555825  146.387431

Random Forest Model 2:
Number of Estimators: 100
Test RMSE: 0.8552713754444304
Test R-squared: 0.9921019352462954
Metrics Table:
      Metric    Training     Testing
0       RMSE    0.356752    0.855271
1  R-squared    0.998802    0.992102
2        MSE    0.127272    0.731489
3        MAE    0.259364    0.653548
4       MAPE    1.615750    2.858417
5        RSS  101.817828  146.297825

Random Forest Model 3:
Number of Estimators: 200
Test RMSE: 0.8547160178085811
Test R-squared: 0.9921121888958909
Metrics Table:
      Metric   Training     Testing
0       RMSE   0.346811    0.854716
1  R-squared   0.99886

### Exercise 3

Say we wish to understand which features have the most impact on crop yield predictions.

Use the `feature_importances_` attribute from our last trained random forest model in **Exercise 2** to return a series containing the feature importance score for each of the features in our dataset, sorted in descending order. 

In [68]:
df_training = df_encoded.drop(columns=[target_column])

In [69]:
# Feature Importance
feature_importances = trained_model.feature_importances_


# Create a DataFrame for feature importances
feature_importances_df = pd.DataFrame({
    'Feature': df_training.columns,
    'Importance': feature_importances
}).sort_values('Importance', ascending=False)

feature_importances_df

,Feature,Importance
1,Rainfall,0.978716
2,Fertilizer_Usage,0.016537
0,Temperature,0.001922
3,Pesticide_Usage,0.001109
12,Crop_Variety_Variety A,0.000301
4,Irrigation,0.000224
13,Crop_Variety_Variety B,0.000195
8,Region_West,0.000178
10,Soil_Type_Loamy,0.000154
11,Soil_Type_Sandy,0.000119


## Solutions

### Exercise 1

In [70]:
def train_rf_model(data, target_variable, n_estimators):

    # Splitting the dataset into features and target variable
    X = data.drop(target_variable, axis=1)  # Features
    y = data[target_variable]  # Target variable

    # Splitting the dataset into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Initializing the RandomForestRegressor with n_estimators
    rf_model = RandomForestRegressor(n_estimators=n_estimators, random_state=42)

    # Training the model on the training set
    rf_model.fit(X_train, y_train)

    # Making predictions on the test set
    y_pred = rf_model.predict(X_test)

    # Evaluating the model
    mse = metrics.mean_squared_error(y_test, y_pred)  # Setting squared=False returns the RMSE
    r2 = metrics.r2_score(y_test, y_pred)
    
    # Return the trained model and its performance metrics
    return rf_model, {'MSE': mse, 'R2': r2}


The function `train_rf_model` is designed to train and evaluate a random forest regression model. 

It takes three parameters: `data`, `target_variable`, and `n_estimators`.

The function returns two items: the trained random forest model `rf_model` and a dictionary containing the evaluation metrics, `mse` and `r2`.

### Exercise 2

In [71]:
# Number of estimators to evaluate
estimators_list = [50, 100, 200]

# Dictionary to store results
results = {}

# Train and evaluate models with different numbers of estimators
for n in estimators_list:
    # Store the entire returned dictionary as the value for each key
    model, metric = train_rf_model(df_encoded, 'Yield', n)
    results[f"{n} trees"] = metric
    
results

{'50 trees': {'MSE': 0.7319371535372017, 'R2': 0.9920970977810426},
 '100 trees': {'MSE': 0.7314891256546079, 'R2': 0.9921019352462954},
 '200 trees': {'MSE': 0.7305394710985587, 'R2': 0.9921121888958909}}

In the code above, we use the previously created function to train and evaluate multiple random forest models, each with a different number of trees (estimators). 

The for loop iterates over each value in `estimators_list`, where it calls the `train_rf_model()` function, passing the required parameters including the current number of estimators `n` as arguments.

The two items returned by the function are stored in separate variables, `model` and `metric`.

The `results` dictionary is then used to store the evaluation metrics for each model trained with a different number of trees. The keys are strings indicating the number of trees, and the values are the dictionary of metrics returned by the function.

### Exercise 3

In [72]:
# Extract feature importances from the model
feature_importances = model.feature_importances_

# Get the names of the features, excluding the target variable 'Yield'
feature_names =df_encoded.drop('Yield', axis=1).columns

# Create a Pandas series 
importances = pd.Series(feature_importances, index=feature_names)

# Sort the feature importances in descending order
sorted_importances = importances.sort_values(ascending=False)
sorted_importances

Rainfall                  0.978673
Fertilizer_Usage          0.016593
Temperature               0.001901
Pesticide_Usage           0.001028
Crop_Variety_Variety A    0.000342
Irrigation                0.000231
Crop_Variety_Variety B    0.000188
Region_West               0.000185
Soil_Type_Loamy           0.000152
Soil_Type_Sandy           0.000132
Crop_Variety_Variety C    0.000119
Region_East               0.000118
Soil_Type_Clayey          0.000114
Region_North              0.000112
Region_South              0.000111
dtype: float64

In the code above, we use the `feature_importances_` attribute of the trained random forest model to extract the importance scores for each feature. 

The variable `feature_names` stores the list of feature names that were used to train the model. This will be used for mapping each importance score to its corresponding feature name.

`importances` is a Pandas series object where each feature's importance score is associated with its name. 

In `sorted_importances`, we get the importances sorted in descending order to get a quick view of the features considered most important by the model.

> Which top two features contribute the most to the model's predictive ability?

Understanding feature importance and the contribution of each variable to the model's predictions offers us an opportunity to streamline our models. This understanding enables us to focus on the most influential features, thereby reducing model complexity without significantly sacrificing performance.

In refining your model, you should consider an experiment: retrain the model using only the subset of features that have demonstrated the highest importance scores. This encourages an exploration into how much we can reduce complexity while maintaining, or even potentially improving, model accuracy.

<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/ExploreAI_logos/EAI_Blue_Dark.png"  style="width:200px";/>
</div>